# Model Validation — AAPL

Rigorous statistical validation of 7 IVS models against market data for **AAPL**.

**Models:** MLP, MLP-log, MLP-log-arb, Conv, Conv-log, Conv-log-arb, Heston  
**Sections:**
0. Config & Load & Align
1. Error Distribution Analysis
2. Signed Bias (Mean-Error Heatmaps)
3. MAPE / Relative Error
4. KS Test Per-Cell
5. Diebold-Mariano Pairwise Significance
6. Backtesting (Exceedance + Kupiec POF + Traffic Light)
7. Regime Conditioning
8. Arbitrage Violation Counting
9. Surface Smoothness

**Prerequisites:** Run `python scripts/run_pipeline.py --ticker AAPL --stages eval heston compare`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
from pathlib import Path
from collections import OrderedDict
from scipy import stats
from scipy.stats import ks_2samp, chi2

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)

# ═══════════════════════════════════════════════
#  CONFIGURATION — change TICKER to reuse
# ═══════════════════════════════════════════════
TICKER = "AAPL"

ROOT = Path("../..") 
EVAL_DIR = ROOT / "artifacts" / "eval" / TICKER
HESTON_DIR = ROOT / "data" / "processed" / "heston" / "surfaces"
OUT = ROOT / "artifacts" / "validation" / TICKER
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "plots").mkdir(exist_ok=True)
(OUT / "tables").mkdir(exist_ok=True)
print(f"Artifacts → {OUT.resolve()}")

# VAE model directories
VAE_DIRS = OrderedDict([
    ("MLP",          EVAL_DIR / "mlp" / "surfaces"),
    ("MLP-log",      EVAL_DIR / "mlp_log" / "surfaces"),
    ("MLP-log-arb",  EVAL_DIR / "mlp_log_arb" / "surfaces"),
    ("Conv",         EVAL_DIR / "conv" / "surfaces"),
    ("Conv-log",     EVAL_DIR / "conv_log" / "surfaces"),
    ("Conv-log-arb", EVAL_DIR / "conv_log_arb" / "surfaces"),
])

COLOURS = {
    "MLP": "#1f77b4", "MLP-log": "#9467bd", "MLP-log-arb": "#d62728",
    "Conv": "#2ca02c", "Conv-log": "#8c564b", "Conv-log-arb": "#e377c2",
    "Heston": "#ff7f0e",
}
MODEL_NAMES = list(COLOURS.keys())

## §0  Load & Align Surfaces

In [ ]:
# ── Load all VAE surfaces ──
vae_surfaces = {}
grid_spec = None
for name, sdir in VAE_DIRS.items():
    if not sdir.exists():
        print(f"⚠  {name}: not found ({sdir}) — skipping")
        continue
    model_s = np.load(sdir / "vae_surfaces.npy")
    market_s = np.load(sdir / "market_surfaces.npy")
    dates_s = pd.to_datetime(pd.read_csv(sdir / "vae_surface_dates.csv")["date"])
    vae_surfaces[name] = (model_s, market_s, dates_s)
    if grid_spec is None:
        with open(sdir / "grid_spec.json") as f:
            grid_spec = json.load(f)
    print(f"✓ {name:<14} {model_s.shape}  ({len(dates_s)} dates)")

# ── Load Heston ──
heston_path = HESTON_DIR / f"{TICKER}_heston_surfaces.npy"
heston_dates_path = HESTON_DIR / f"{TICKER}_heston_surface_dates.csv"
has_heston = heston_path.exists()
if has_heston:
    heston_surf = np.load(heston_path)
    heston_dates = pd.to_datetime(pd.read_csv(heston_dates_path)["date"])
    print(f"✓ {'Heston':<14} {heston_surf.shape}  ({len(heston_dates)} dates)")
else:
    print("⚠ Heston surfaces not found — will validate VAE models only")

days_grid = np.array(grid_spec["days_grid"])
delta_grid = np.array(grid_spec["delta_grid"])
cp_labels = grid_spec["cp_order"]
n_chan, n_mat, n_del = len(cp_labels), len(days_grid), len(delta_grid)
print(f"\nGrid: {cp_labels} × {n_mat} maturities × {n_del} deltas")

In [ ]:
# ── Align to common dates ──
def _to_date_set(ds):
    return set(ds.dt.date)

first_name = next(iter(vae_surfaces))
common = _to_date_set(vae_surfaces[first_name][2])
for name, (_, _, dates) in vae_surfaces.items():
    common &= _to_date_set(dates)
if has_heston:
    common &= _to_date_set(heston_dates)
common = sorted(common)
N = len(common)

# Build aligned arrays — shape (N, 2, 11, 17)
models = OrderedDict()
for name, (surf, _, dates) in vae_surfaces.items():
    mask = [d in common for d in dates.dt.date]
    models[name] = surf[mask]

if has_heston:
    heston_mask = [d in common for d in heston_dates.dt.date]
    models["Heston"] = heston_surf[heston_mask]

first_vae = next(iter(vae_surfaces.values()))
market_mask = [d in common for d in first_vae[2].dt.date]
market = first_vae[1][market_mask]
aligned_dates = first_vae[2][market_mask].reset_index(drop=True)

# Use only models that were actually loaded
MODEL_NAMES = [n for n in MODEL_NAMES if n in models]

# Valid mask (handle Heston NaNs)
valid_mask = np.isfinite(market)
for name, surf in models.items():
    valid_mask &= np.isfinite(surf)

pct_valid = 100 * valid_mask.sum() / valid_mask.size
print(f"Common dates : {N}")
print(f"Date range   : {common[0]}  →  {common[-1]}")
print(f"Valid cells  : {valid_mask.sum():,} / {valid_mask.size:,} ({pct_valid:.1f}%)")

# Precompute errors
errors = OrderedDict()
abs_errors = OrderedDict()
for name, surf in models.items():
    e = surf - market
    err_flat = e[valid_mask]
    errors[name] = err_flat
    abs_errors[name] = np.abs(err_flat)
    print(f"  {name:<14} MAE={np.abs(err_flat).mean()*100:.2f} vp   "
          f"RMSE={np.sqrt((err_flat**2).mean())*100:.2f} vp")

## §1  Error Distribution Analysis

In [ ]:
# ── 1a. Signed-error histograms ──
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4*len(MODEL_NAMES), 4), sharey=True)
if len(MODEL_NAMES) == 1:
    axes = [axes]
for ax, name in zip(axes, MODEL_NAMES):
    e = errors[name] * 100
    ax.hist(e, bins=80, color=COLOURS[name], alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Error (vol pts)")
    ax.set_xlim(-15, 15)
axes[0].set_ylabel("Count")
fig.suptitle(f"{TICKER} — Signed Error Distribution", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "plots" / "error_histograms.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── 1b. QQ-plots ──
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4*len(MODEL_NAMES), 4))
if len(MODEL_NAMES) == 1:
    axes = [axes]
for ax, name in zip(axes, MODEL_NAMES):
    e = errors[name] * 100
    stats.probplot(e, dist="norm", plot=ax)
    ax.set_title(name, fontsize=11)
    ax.get_lines()[0].set(color=COLOURS[name], markersize=1.5, alpha=0.5)
    ax.get_lines()[1].set(color="red", lw=1)
fig.suptitle(f"{TICKER} — QQ-Plot vs Normal", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "plots" / "error_qq_plots.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── 1c. Percentile table ──
percentiles = [50, 75, 90, 95, 99]
rows = []
for name in MODEL_NAMES:
    ae = abs_errors[name] * 100
    se = errors[name] * 100
    row = {"Model": name, "Mean": se.mean(), "Std": se.std(),
           "Skew": float(stats.skew(se)), "Kurt": float(stats.kurtosis(se))}
    for p in percentiles:
        row[f"P{p}"] = np.percentile(ae, p)
    row["Max"] = ae.max()
    rows.append(row)

err_table = pd.DataFrame(rows).set_index("Model")
err_table.to_csv(OUT / "tables" / "error_distribution_stats.csv")
print(f"{TICKER} — Error Distribution (vol points):")
display(err_table.round(3))

## §2  Signed Bias (Mean-Error Heatmaps)

In [ ]:
delta_labels = [f"{d:.0f}" for d in delta_grid * 100]
mat_labels = [f"{d:.0f}d" for d in days_grid]

for ch_idx, ch_name in enumerate(cp_labels):
    fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4*len(MODEL_NAMES), 4), sharey=True)
    if len(MODEL_NAMES) == 1:
        axes = [axes]
    vmax = 0
    mean_errs = {}
    for name in MODEL_NAMES:
        err_grid = models[name][:, ch_idx, :, :] - market[:, ch_idx, :, :]
        cell_valid = valid_mask[:, ch_idx, :, :]
        with np.errstate(invalid="ignore"):
            me = np.where(cell_valid, err_grid, np.nan)
            mean_errs[name] = np.nanmean(me, axis=0) * 100
        vmax = max(vmax, np.nanmax(np.abs(mean_errs[name])))

    for ax, name in zip(axes, MODEL_NAMES):
        im = ax.imshow(mean_errs[name], cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                        aspect="auto", origin="lower")
        ax.set_title(name, fontsize=10)
        ax.set_xticks(range(n_del))
        ax.set_xticklabels(delta_labels, fontsize=5, rotation=45)
        ax.set_xlabel("Delta")
        if ax is axes[0]:
            ax.set_yticks(range(n_mat))
            ax.set_yticklabels(mat_labels, fontsize=7)
            ax.set_ylabel("Maturity")
        else:
            ax.set_yticks([])

    fig.colorbar(im, ax=axes, shrink=0.8, label="Mean Error (vol pts)")
    fig.suptitle(f"{TICKER} — Mean Signed Error — {ch_name}", fontsize=13, y=1.03)
    fig.tight_layout()
    fig.savefig(OUT / "plots" / f"bias_heatmap_{ch_name}.png", bbox_inches="tight", dpi=150)
    plt.show()

## §3  MAPE / Relative Error

In [ ]:
def mape_grid(model_surf, market_surf, vmask):
    with np.errstate(divide="ignore", invalid="ignore"):
        pct = np.abs(model_surf - market_surf) / np.abs(market_surf) * 100
        pct = np.where(vmask, pct, np.nan)
    return np.nanmean(pct, axis=0)

fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4*len(MODEL_NAMES), 4), sharey=True)
if len(MODEL_NAMES) == 1:
    axes = [axes]
vmax_mape = 0
mape_grids = {}
for name in MODEL_NAMES:
    mg = mape_grid(models[name], market, valid_mask)
    mape_grids[name] = mg.mean(axis=0)
    vmax_mape = max(vmax_mape, np.nanmax(mape_grids[name]))

for ax, name in zip(axes, MODEL_NAMES):
    im = ax.imshow(mape_grids[name], cmap="YlOrRd", vmin=0, vmax=min(vmax_mape, 30),
                    aspect="auto", origin="lower")
    ax.set_title(name, fontsize=10)
    ax.set_xticks(range(n_del))
    ax.set_xticklabels(delta_labels, fontsize=5, rotation=45)
    ax.set_xlabel("Delta")
    if ax is axes[0]:
        ax.set_yticks(range(n_mat))
        ax.set_yticklabels(mat_labels, fontsize=7)
        ax.set_ylabel("Maturity")
    else:
        ax.set_yticks([])

fig.colorbar(im, ax=axes, shrink=0.8, label="MAPE (%)")
fig.suptitle(f"{TICKER} — MAPE per Grid Cell", fontsize=13, y=1.03)
fig.tight_layout()
fig.savefig(OUT / "plots" / "mape_heatmaps.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# MAPE summary: overall vs high/low IV
rows = []
for name in MODEL_NAMES:
    m_flat = market[valid_mask]
    s_flat = models[name][valid_mask]
    pct_all = np.abs(s_flat - m_flat) / np.abs(m_flat) * 100
    high_mask = m_flat > 0.20
    low_mask = m_flat <= 0.20
    rows.append({
        "Model": name,
        "MAPE (all)": pct_all.mean(),
        "MAPE (IV>20%)": pct_all[high_mask].mean() if high_mask.sum() > 0 else np.nan,
        "MAPE (IV≤20%)": pct_all[low_mask].mean() if low_mask.sum() > 0 else np.nan,
        "N (IV>20%)": int(high_mask.sum()),
        "N (IV≤20%)": int(low_mask.sum()),
    })

mape_table = pd.DataFrame(rows).set_index("Model")
mape_table.to_csv(OUT / "tables" / "mape_summary.csv")
display(mape_table.round(2))

## §4  KS Test Per-Cell

In [ ]:
ks_pvals = {}
ks_stats_arr = {}

for name in MODEL_NAMES:
    pvals = np.full((n_chan, n_mat, n_del), np.nan)
    kstats = np.full((n_chan, n_mat, n_del), np.nan)
    surf = models[name]
    for c in range(n_chan):
        for h in range(n_mat):
            for w in range(n_del):
                vmask_cell = valid_mask[:, c, h, w]
                if vmask_cell.sum() < 10:
                    continue
                m_vals = market[vmask_cell, c, h, w]
                s_vals = surf[vmask_cell, c, h, w]
                stat, p = ks_2samp(m_vals, s_vals)
                kstats[c, h, w] = stat
                pvals[c, h, w] = p
    ks_pvals[name] = pvals
    ks_stats_arr[name] = kstats
    reject_pct = 100 * np.nansum(pvals < 0.05) / np.sum(~np.isnan(pvals))
    print(f"{name:<14} reject (p<0.05): {np.nansum(pvals < 0.05):3.0f} / "
          f"{np.sum(~np.isnan(pvals))} ({reject_pct:.1f}%)")

In [ ]:
# KS p-value heatmaps
for ch_idx, ch_name in enumerate(cp_labels):
    fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4*len(MODEL_NAMES), 4), sharey=True)
    if len(MODEL_NAMES) == 1:
        axes = [axes]
    for ax, name in zip(axes, MODEL_NAMES):
        pv = ks_pvals[name][ch_idx]
        log_p = -np.log10(np.clip(pv, 1e-20, 1.0))
        im = ax.imshow(log_p, cmap="YlOrRd", vmin=0, vmax=5, aspect="auto", origin="lower")
        for h in range(n_mat):
            for w in range(n_del):
                if pv[h, w] < 0.05:
                    ax.text(w, h, "×", ha="center", va="center", fontsize=6, color="blue", fontweight="bold")
        ax.set_title(name, fontsize=10)
        ax.set_xticks(range(n_del))
        ax.set_xticklabels(delta_labels, fontsize=5, rotation=45)
        ax.set_xlabel("Delta")
        if ax is axes[0]:
            ax.set_yticks(range(n_mat))
            ax.set_yticklabels(mat_labels, fontsize=7)
            ax.set_ylabel("Maturity")
        else:
            ax.set_yticks([])

    fig.colorbar(im, ax=axes, shrink=0.8, label="$-\\log_{10}(p)$")
    fig.suptitle(f"{TICKER} — KS p-values — {ch_name}  (× = reject at 5%)", fontsize=13, y=1.03)
    fig.tight_layout()
    fig.savefig(OUT / "plots" / f"ks_pvalues_{ch_name}.png", bbox_inches="tight", dpi=150)
    plt.show()

# Summary table
ks_summary = []
for name in MODEL_NAMES:
    pv = ks_pvals[name]
    valid_cells = np.sum(~np.isnan(pv))
    ks_summary.append({
        "Model": name,
        "Cells tested": int(valid_cells),
        "Reject (p<0.05)": int(np.nansum(pv < 0.05)),
        "Reject (p<0.01)": int(np.nansum(pv < 0.01)),
        "Reject %": 100 * np.nansum(pv < 0.05) / valid_cells,
        "Median p": np.nanmedian(pv),
        "Mean KS stat": np.nanmean(ks_stats_arr[name]),
    })
ks_df = pd.DataFrame(ks_summary).set_index("Model")
ks_df.to_csv(OUT / "tables" / "ks_test_summary.csv")
display(ks_df.round(4))

## §5  Diebold-Mariano Pairwise Significance

In [ ]:
def diebold_mariano(e1, e2):
    d = e1**2 - e2**2
    d_bar = d.mean()
    n = len(d)
    gamma_0 = np.var(d, ddof=1)
    se = np.sqrt(gamma_0 / n)
    if se < 1e-15:
        return 0.0, 1.0
    t = d_bar / se
    p = 2 * (1 - stats.norm.cdf(np.abs(t)))
    return t, p

# Per-date MSE
date_mse = {}
for name in MODEL_NAMES:
    err = models[name] - market
    se_grid = np.where(valid_mask, err**2, np.nan)
    date_mse[name] = np.nanmean(se_grid.reshape(N, -1), axis=1)

n_models = len(MODEL_NAMES)
dm_tstat = np.zeros((n_models, n_models))
dm_pval = np.ones((n_models, n_models))

for i in range(n_models):
    for j in range(n_models):
        if i == j:
            continue
        e1 = np.sqrt(date_mse[MODEL_NAMES[i]])
        e2 = np.sqrt(date_mse[MODEL_NAMES[j]])
        t, p = diebold_mariano(e1, e2)
        dm_tstat[i, j] = t
        dm_pval[i, j] = p

dm_t_df = pd.DataFrame(dm_tstat, index=MODEL_NAMES, columns=MODEL_NAMES)
dm_p_df = pd.DataFrame(dm_pval, index=MODEL_NAMES, columns=MODEL_NAMES)

print("DM t-statistics (positive → row model is worse):")
display(dm_t_df.round(3))
print("\nDM p-values (two-sided):")
display(dm_p_df.round(4))

# Heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
im1 = ax1.imshow(dm_tstat, cmap="RdBu_r", vmin=-5, vmax=5)
ax1.set_xticks(range(n_models)); ax1.set_xticklabels(MODEL_NAMES, rotation=45, ha="right")
ax1.set_yticks(range(n_models)); ax1.set_yticklabels(MODEL_NAMES)
for i in range(n_models):
    for j in range(n_models):
        ax1.text(j, i, f"{dm_tstat[i,j]:.2f}", ha="center", va="center", fontsize=7)
ax1.set_title("DM t-statistic")
fig.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(-np.log10(np.clip(dm_pval, 1e-20, 1)), cmap="YlOrRd", vmin=0, vmax=5)
ax2.set_xticks(range(n_models)); ax2.set_xticklabels(MODEL_NAMES, rotation=45, ha="right")
ax2.set_yticks(range(n_models)); ax2.set_yticklabels(MODEL_NAMES)
for i in range(n_models):
    for j in range(n_models):
        marker = ""
        if dm_pval[i,j] < 0.01: marker = "***"
        elif dm_pval[i,j] < 0.05: marker = "**"
        elif dm_pval[i,j] < 0.10: marker = "*"
        ax2.text(j, i, f"{dm_pval[i,j]:.3f}\n{marker}", ha="center", va="center", fontsize=6)
ax2.set_title("$-\\log_{10}(p)$  (* p<0.10, ** p<0.05, *** p<0.01)")
fig.colorbar(im2, ax=ax2, shrink=0.8)

fig.suptitle(f"{TICKER} — Diebold-Mariano Pairwise Test", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(OUT / "plots" / "diebold_mariano.png", bbox_inches="tight", dpi=150)
plt.show()

dm_t_df.to_csv(OUT / "tables" / "dm_tstat.csv")
dm_p_df.to_csv(OUT / "tables" / "dm_pval.csv")

## §6  Backtesting (Exceedance + Kupiec POF + Traffic Light)

In [ ]:
def kupiec_pof(T, N_exc, alpha):
    p_hat = N_exc / T
    if N_exc == 0:
        lr = -2 * T * np.log(1 - alpha)
        return lr, 1 - chi2.cdf(lr, 1)
    if N_exc == T:
        lr = -2 * T * np.log(alpha)
        return lr, 1 - chi2.cdf(lr, 1)
    lr = -2 * (N_exc * np.log(alpha / p_hat) + (T - N_exc) * np.log((1 - alpha) / (1 - p_hat)))
    return lr, 1 - chi2.cdf(lr, 1)

def traffic_light(N_exc, T, alpha):
    expected = T * alpha
    std = np.sqrt(T * alpha * (1 - alpha))
    if N_exc <= expected + std:
        return "🟢 Green"
    elif N_exc <= expected + 2 * std:
        return "🟡 Yellow"
    else:
        return "🔴 Red"

# Quantile-based backtesting
alphas = [0.01, 0.025, 0.05]
qt_rows = []
for name in MODEL_NAMES:
    date_mae = np.array([
        np.nanmean(np.abs(models[name][t][valid_mask[t]] - market[t][valid_mask[t]]))
        for t in range(N)
    ])
    for alpha in alphas:
        thr = np.quantile(date_mae, 1 - alpha)
        N_exc = (date_mae > thr).sum()
        expected = N * alpha
        lr, pval = kupiec_pof(N, N_exc, alpha)
        tl = traffic_light(N_exc, N, alpha)
        qt_rows.append({
            "Model": name, "α": f"{alpha:.1%}",
            "Threshold (vp)": thr * 100,
            "Exceedances": N_exc, "Expected": expected,
            "Kupiec LR": lr, "Kupiec p": pval,
            "Traffic Light": tl,
        })

qt_df = pd.DataFrame(qt_rows)
display(qt_df.set_index(["Model", "α"]))
qt_df.to_csv(OUT / "tables" / "backtest_quantile.csv", index=False)

In [ ]:
# Fixed-threshold backtesting
thresholds_vp = [2, 5, 10]
ft_rows = []
for name in MODEL_NAMES:
    date_mae = np.array([
        np.nanmean(np.abs(models[name][t][valid_mask[t]] - market[t][valid_mask[t]]))
        for t in range(N)
    ]) * 100
    for thr in thresholds_vp:
        N_exc = (date_mae > thr).sum()
        ft_rows.append({
            "Model": name, "Threshold (vp)": thr,
            "Exceedances": N_exc, "Rate": f"{N_exc/N:.1%}",
        })

ft_df = pd.DataFrame(ft_rows)
display(ft_df.set_index(["Model", "Threshold (vp)"]))
ft_df.to_csv(OUT / "tables" / "backtest_fixed_threshold.csv", index=False)

# Visual
fig, ax = plt.subplots(figsize=(14, 5))
for name in MODEL_NAMES:
    date_mae = np.array([
        np.nanmean(np.abs(models[name][t][valid_mask[t]] - market[t][valid_mask[t]]))
        for t in range(N)
    ]) * 100
    ax.plot(aligned_dates, date_mae, color=COLOURS[name], label=name, alpha=0.8, lw=1)
for thr in thresholds_vp:
    ax.axhline(thr, color="grey", ls="--", lw=0.7, alpha=0.5)
    ax.text(aligned_dates.iloc[-1], thr + 0.2, f"{thr} vp", fontsize=8, color="grey")
ax.set_ylabel("Mean Abs Error (vol pts)")
ax.set_xlabel("Date")
ax.set_title(f"{TICKER} — Per-Date MAE with Fixed Thresholds")
ax.legend(fontsize=8, ncol=4, loc="upper left")
fig.tight_layout()
fig.savefig(OUT / "plots" / "backtest_timeseries.png", bbox_inches="tight", dpi=150)
plt.show()

## §7  Regime Conditioning

In [ ]:
# ATM IV proxy
atm_delta_idx = np.argmin(np.abs(delta_grid - 0.50))
mid_mat_idx = np.argmin(np.abs(days_grid - 182))
print(f"ATM proxy: delta[{atm_delta_idx}]={delta_grid[atm_delta_idx]:.2f}, "
      f"maturity[{mid_mat_idx}]={days_grid[mid_mat_idx]:.0f}d")

atm_iv = market[:, 0, mid_mat_idx, atm_delta_idx]
print(f"ATM IV range: [{atm_iv.min()*100:.1f}%, {atm_iv.max()*100:.1f}%]")

t1, t2 = np.percentile(atm_iv, [33.3, 66.7])
regime_labels = np.where(atm_iv <= t1, "Low", np.where(atm_iv <= t2, "Mid", "High"))
print(f"Terciles: Low ≤ {t1*100:.1f}%, Mid ≤ {t2*100:.1f}%, High > {t2*100:.1f}%")
print(f"Counts: Low={np.sum(regime_labels=='Low')}, Mid={np.sum(regime_labels=='Mid')}, "
      f"High={np.sum(regime_labels=='High')}")

In [ ]:
regime_rows = []
for name in MODEL_NAMES:
    for regime in ["Low", "Mid", "High"]:
        idx = np.where(regime_labels == regime)[0]
        surf_sub = models[name][idx]
        mkt_sub = market[idx]
        vm_sub = valid_mask[idx]
        ae = np.abs(surf_sub[vm_sub] - mkt_sub[vm_sub])
        regime_rows.append({
            "Model": name, "Regime": regime,
            "MAE (vp)": ae.mean() * 100,
            "RMSE (vp)": np.sqrt((ae**2).mean()) * 100,
            "P95 (vp)": np.percentile(ae, 95) * 100,
            "N dates": len(idx),
        })

regime_df = pd.DataFrame(regime_rows)
regime_pivot = regime_df.pivot_table(
    index="Model", columns="Regime",
    values=["MAE (vp)", "RMSE (vp)"],
)[["MAE (vp)", "RMSE (vp)"]].round(3)
regime_pivot.to_csv(OUT / "tables" / "regime_mae.csv")
display(regime_pivot)

# Scatter
fig, ax = plt.subplots(figsize=(12, 5))
for name in MODEL_NAMES:
    date_mae = np.array([
        np.nanmean(np.abs(models[name][t][valid_mask[t]] - market[t][valid_mask[t]]))
        for t in range(N)
    ]) * 100
    ax.scatter(atm_iv * 100, date_mae, color=COLOURS[name], label=name, alpha=0.5, s=15, edgecolors="none")
ax.set_xlabel("ATM IV (%)")
ax.set_ylabel("Date MAE (vol pts)")
ax.set_title(f"{TICKER} — Model Error vs Market Volatility Level")
ax.axvline(t1 * 100, color="grey", ls=":", lw=0.8)
ax.axvline(t2 * 100, color="grey", ls=":", lw=0.8)
ax.legend(fontsize=8, ncol=4)
fig.tight_layout()
fig.savefig(OUT / "plots" / "regime_scatter.png", bbox_inches="tight", dpi=150)
plt.show()

## §8  Arbitrage Violation Counting

In [ ]:
def count_calendar_violations(surfaces, days, valid=None):
    T = np.array(days) / 365.0
    T = T[np.newaxis, np.newaxis, :, np.newaxis]
    total_var = surfaces**2 * T
    diff = np.diff(total_var, axis=2)
    if valid is not None:
        v1 = valid[:, :, :-1, :]
        v2 = valid[:, :, 1:, :]
        vmask = v1 & v2
        violations = (diff[vmask] < -1e-8)
    else:
        violations = (diff < -1e-8)
    return int(violations.sum()), violations

def count_butterfly_violations(surfaces, valid=None):
    d2 = np.diff(surfaces, n=2, axis=3)
    if valid is not None:
        v1 = valid[:, :, :, :-2]
        v2 = valid[:, :, :, 1:-1]
        v3 = valid[:, :, :, 2:]
        vmask = v1 & v2 & v3
        violations = (d2[vmask] < -1e-8)
    else:
        violations = (d2 < -1e-8)
    return int(violations.sum()), violations

arb_rows = []
all_surfs = {**models, "Market": market}
for name, surf in all_surfs.items():
    vm = valid_mask if name != "Market" else np.ones_like(valid_mask)
    cal_count, _ = count_calendar_violations(surf, days_grid, vm)
    but_count, _ = count_butterfly_violations(surf, vm)
    cal_total = int(vm[:, :, :-1, :].sum()) if name != "Market" else N * n_chan * (n_mat - 1) * n_del
    but_total = int(vm[:, :, :, :-2].sum()) if name != "Market" else N * n_chan * n_mat * (n_del - 2)
    arb_rows.append({
        "Surface": name,
        "Calendar violations": cal_count,
        "Calendar total": cal_total,
        "Calendar %": 100 * cal_count / max(cal_total, 1),
        "Butterfly violations": but_count,
        "Butterfly total": but_total,
        "Butterfly %": 100 * but_count / max(but_total, 1),
    })

arb_df = pd.DataFrame(arb_rows).set_index("Surface")
arb_df.to_csv(OUT / "tables" / "arbitrage_violations.csv")
display(arb_df.round(3))

## §9  Surface Smoothness

In [ ]:
def laplacian_norm(surfaces):
    lap_mat = np.diff(surfaces, n=2, axis=2)
    lap_del = np.diff(surfaces, n=2, axis=3)
    return np.nanmean(np.abs(lap_mat)) + np.nanmean(np.abs(lap_del))

def total_variation(surfaces):
    dx = np.diff(surfaces, axis=2)
    dy = np.diff(surfaces, axis=3)
    return np.nanmean(np.abs(dx)) + np.nanmean(np.abs(dy))

smooth_rows = []
all_surfs_smooth = {**models, "Market": market}
for name, surf in all_surfs_smooth.items():
    smooth_rows.append({"Surface": name, "Laplacian |∇²σ|": laplacian_norm(surf),
                        "Total Variation": total_variation(surf)})

smooth_df = pd.DataFrame(smooth_rows).set_index("Surface").sort_values("Laplacian |∇²σ|")
smooth_df.to_csv(OUT / "tables" / "smoothness.csv")
print(f"{TICKER} — Surface Smoothness (lower = smoother):")
display(smooth_df.round(6))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
names = smooth_df.index.tolist()
colors = [COLOURS.get(n, "grey") for n in names]
ax1.barh(names, smooth_df["Laplacian |∇²σ|"], color=colors)
ax1.set_xlabel("Mean |∇²σ|")
ax1.set_title("Laplacian Norm")
ax2.barh(names, smooth_df["Total Variation"], color=colors)
ax2.set_xlabel("Mean TV")
ax2.set_title("Total Variation")
fig.suptitle(f"{TICKER} — Surface Smoothness (lower = smoother)")
fig.tight_layout()
fig.savefig(OUT / "plots" / "smoothness.png", bbox_inches="tight", dpi=150)
plt.show()

## Summary

All validation artifacts saved to `artifacts/validation/{TICKER}/`:
- `plots/` — PNG figures
- `tables/` — CSV tables

**Key outputs:**
| File | Content |
|------|---------|
| `error_distribution_stats.csv` | Percentile table (P50–P99, max, skew, kurtosis) |
| `mape_summary.csv` | MAPE overall vs high/low IV subsets |
| `ks_test_summary.csv` | KS rejection rates per model |
| `dm_tstat.csv`, `dm_pval.csv` | Diebold-Mariano pairwise matrices |
| `backtest_quantile.csv` | Kupiec POF + traffic light |
| `backtest_fixed_threshold.csv` | Fixed threshold exceedance rates |
| `regime_mae.csv` | MAE by vol regime tercile |
| `arbitrage_violations.csv` | Calendar + butterfly violation counts |
| `smoothness.csv` | Laplacian + TV rankings |